In [86]:
import pandas as pd
import sys
from pathlib import Path

In [87]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

In [88]:
from src.load_data import load_data

In [89]:
df = load_data()

In [90]:
from sklearn.model_selection import train_test_split

In [100]:
X = df.drop("y", axis=1)
y = df["y"].map({"no": 0, "yes": 1})

In [101]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [102]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [103]:
from sklearn.dummy import DummyClassifier

In [104]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(x_train, y_train)
y_pred = dummy_model.predict(x_test)

In [105]:
df["y"].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [106]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred, pos_label='1'))
print("Recall: ", recall_score(y_test, y_pred, pos_label='1'))
print("F1 Score: ", f1_score(y_test, y_pred, pos_label='1'))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred, labels=[0, 1]))

Accuracy:  0.8793541966161672


ValueError: pos_label=1 is not a valid label. It should be one of [0, 1]

In [ ]:
baseline_results ={
    "Model": "Dummy Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, pos_label='1'),
    "Recall": recall_score(y_test, y_pred, pos_label='1'),
    "F1 Score": f1_score(y_test, y_pred, pos_label='1'),
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
}
baseline_results_df = pd.DataFrame([baseline_results])
print(baseline_results_df)

              Model  Accuracy  Precision  Recall  F1 Score  \
0  Dummy Classifier  0.879354        0.0     0.0       0.0   

         confusion_matrix  
0  [[7952, 0], [1091, 0]]  


c:\Users\deniz\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [23]:
baseline_results_df.to_csv(ROOT_DIR / "reports" / "baseline_results.csv", index=False)

In [65]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [66]:
categorical_features = ["job", "marital", "education", "contact"]
numerical_features = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

In [67]:
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [68]:
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

In [69]:
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features)
])

In [70]:
x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed = preprocessor.transform(x_val)

In [71]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


In [72]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "CatBoost": CatBoostClassifier()
}

In [73]:
result = []

In [74]:
for model_name, model in models.items():
    model.fit(x_train_processed, y_train)
    y_pred = model.predict(x_val_processed)

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, pos_label='yes', zero_division=0)
    recall = recall_score(y_val, y_pred, pos_label='yes', zero_division=0)
    f1 = f1_score(y_val, y_pred, pos_label='yes', zero_division=0)
    conf_matrix = confusion_matrix(y_val, y_pred, labels=['no', 'yes']).tolist()

    result.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['no' 'yes']